# SPIN 7B — Step 3 (AUTHOR-SAMPLE, provisoire): generate SALAD answers

Throwaway A/B copy of `7B testing/SPIN_7B_step3_generate.ipynb`. **Identical generation code.**
Differences, so it never collides with the 7B outputs:
- reads **`coupled_indices_authorsample.json`**,
- tags outputs **`*_authorsample`** -> `answers_Qwen2-7B-Instruct_authorsample_{fairness,privacy}.csv`,
  `results_Qwen2-7B-Instruct_authorsample.json`,
- separate checkpoint `gen_ckpt_authorsample.json`.

Add `coupled_indices_authorsample.json` as input, run (~3-5 h full SALAD, resumable), then feed the
CSVs to the Step 4 author-sample judge.


## Step 0 — env + config + coupled indices

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, json, gc, time
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
N_EVAL=None                # None = full SALAD sets (authors' setting). Set an int to subsample.
USE_CHAT_TEMPLATE=False    # authors feed RAW questions (verified)
GEN_BS=4
MAXNEW=64                  # eval_salad SamplingParams(max_tokens=64)
DTYPE=torch.float16

def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
CP=find('coupled_indices_authorsample.json'); assert CP, 'add coupled_indices_authorsample.json (author-sample Step 1) as input'
COUPLED=json.load(open(CP))['coupled']
print('coupled matrices', len(COUPLED), '| coupled weights', sum(len(v) for v in COUPLED.values()))

## Step 1 — data + model

In [ ]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
fair_q=pd.read_csv(f'{DATA}/salad_fairness.csv')['questions'].tolist()[:N_EVAL]
priv_q=pd.read_csv(f'{DATA}/salad_privacy.csv')['questions'].tolist()[:N_EVAL]

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model split loaded | entry', GEN_DEV, '|', len(fair_q), 'fair +', len(priv_q), 'priv')

def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
def suppress(coupled):
    d=dict(model.named_parameters()); nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); p.data.view(-1)[idx]=0.0; nz+=len(idxlist)
    return nz

## Step 2 — generate (checkpointed, resumable)

In [ ]:
CK='/kaggle/working/gen_ckpt_authorsample.json'
state=json.load(open(CK)) if os.path.exists(CK) else {}

@torch.no_grad()
def gen_set(key, queries):
    done=state.get(key, []); t0=time.time()
    for i in range(len(done), len(queries), GEN_BS):
        enc=tokenizer(fmt(queries[i:i+GEN_BS]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        g=model.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        done+=[tokenizer.decode(g[j,enc.input_ids.shape[1]:],skip_special_tokens=True).strip() for j in range(g.shape[0])]
        state[key]=done; json.dump(state, open(CK,'w'))
        print(f'    {key} {len(done)}/{len(queries)} ({time.time()-t0:.0f}s)', end='\r')
    print(); return done

# baseline (original model)
print('baseline generation...')
base_fair=gen_set('base_fair', fair_q); base_priv=gen_set('base_priv', priv_q)
# apply SPIN, then generate again (on resume, base sets are already done and skipped instantly)
nz=suppress(COUPLED); print('applied SPIN: zeroed', nz)
print('SPIN generation...')
spin_fair=gen_set('spin_fair', fair_q); spin_priv=gen_set('spin_priv', priv_q)

tag=MODEL_ID.split('/')[-1]+'_authorsample'
pd.DataFrame({'q':fair_q,'baseline':base_fair,'spin':spin_fair}).to_csv(f'/kaggle/working/answers_{tag}_fairness.csv',index=False)
pd.DataFrame({'q':priv_q,'baseline':base_priv,'spin':spin_priv}).to_csv(f'/kaggle/working/answers_{tag}_privacy.csv',index=False)
json.dump({'model':MODEL_ID,'tag':tag,'coupled_zeroed':int(nz),'n_eval':N_EVAL},
          open(f'/kaggle/working/results_{tag}.json','w'), indent=2)
print(f'\nSAVED answers_{tag}_*.csv + results_{tag}.json')
print('Next: score with the Step 4 author-sample judge (add these CSVs + results JSON as input).')